In [ ]:
# AGGREGATED MORAN PPC — ONE STEP AHEAD (5 MODELS)
# IID / BYM / BYM weekly / BYM+cov / BYM weekly+cov
# Condition on observed y_{t-1}
import numpy as np
import pickle
import pyreadr
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

np.random.seed(1)
# Paths and settings
BASE_DIR = Path(r"path/to/snow/data-and-results")
period = 52

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1
# Data
snow = pyreadr.read_r(BASE_DIR / "snow_cleaned_full.Rda")
snow = list(snow.values())[0]

coords = snow.iloc[:, :2].to_numpy()
y_full = snow.iloc[:, 2:].to_numpy()

coords = np.delete(coords, no_nbs, axis=0)
y_full = np.delete(y_full, no_nbs, axis=0)
# TWO LARGEST COMPONENTS
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords[:,0], coords[:,1]),
    crs="EPSG:4326"
).to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")

xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6
D = squareform(pdist(xy))
W_full = (D <= 0.22).astype(int)
np.fill_diagonal(W_full, 0)
W_full = csr_matrix(W_full)

n_comp, labels = connected_components(W_full, directed=False)
sizes = np.bincount(labels)
order = np.argsort(sizes)[::-1]

keep = np.sort(np.concatenate([
    np.where(labels == order[0])[0],
    np.where(labels == order[1])[0]
]))

coords = coords[keep]
y = y_full[keep]

S, T = y.shape

# rebuild adjacency
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords[:,0], coords[:,1]),
    crs="EPSG:4326"
).to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")

xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6
D = squareform(pdist(xy))
W = (D <= 0.22).astype(int)
np.fill_diagonal(W, 0)
W = csr_matrix(W)
w_sum = W.sum()

print("Using S =", S)
# OBSERVED MORAN (aggregated)
y_bar = y[:,1:].mean(axis=1)
yc = y_bar - y_bar.mean()
I_obs = (S/w_sum)*((yc @ (W @ yc))/(yc@yc))
print("Observed Moran:", I_obs)
# TIME COVARIATES
t = np.arange(1,T+1)
week = np.arange(T) % 52
t_trend = (t - t.mean())/t.std(ddof=0)

cov4 = np.column_stack([
    np.ones(T),
    np.cos(2*np.pi*t/period),
    np.sin(2*np.pi*t/period),
    t_trend
])

cov8 = np.column_stack([
    np.ones(T), np.ones(T),
    np.cos(2*np.pi*t/period), np.cos(2*np.pi*t/period),
    np.sin(2*np.pi*t/period), np.sin(2*np.pi*t/period),
    t_trend, t_trend
])
# SPATIAL COVARIATES
lat = (coords[:,1] - coords[:,1].mean()) / coords[:,1].std()

elev = pd.read_csv(BASE_DIR/"curr_elev.csv").iloc[:,3].to_numpy()[keep]
elev = (elev - elev.mean())/elev.std()

snow_temp = pyreadr.read_r(BASE_DIR/"snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)
temp = snow_temp.drop(index=no_nbs).iloc[:,2:].to_numpy()[keep]
temp = (temp - temp.mean())/temp.std()
# LOAD POSTERIORS
thin = 15
n_chains = 10

iid01_list = []
iid10_list = []

for c in range(n_chains):
    d01 = np.load(BASE_DIR/f"ind01_chain{c}.npz")
    d10 = np.load(BASE_DIR/f"ind10_chain{c}.npz")

    iid01_list.append(d01["theta"][:, ::thin])
    iid10_list.append(d10["theta"][:, ::thin])

iid01 = np.concatenate(iid01_list, axis=1)
iid10 = np.concatenate(iid10_list, axis=1)



eta01_list = []
tau01_list = []
eta10_list = []
tau10_list = []

for c in range(10):

    with open(BASE_DIR/f"bym01_chain{c}.pkl","rb") as f:
        d = pickle.load(f)
    eta01_list.append(d["eta"][:, ::15])
    tau01_list.append(d["tau"][:, ::15])

    with open(BASE_DIR/f"bym10_chain{c}.pkl","rb") as f:
        d = pickle.load(f)
    eta10_list.append(d["eta"][:, ::15])
    tau10_list.append(d["tau"][:, ::15])

eta01_week = np.concatenate(eta01_list, axis=1)
tau01_week = np.concatenate(tau01_list, axis=1)
eta10_week = np.concatenate(eta10_list, axis=1)
tau10_week = np.concatenate(tau10_list, axis=1)



eta01_wf_list = []
tau01_wf_list = []
eta10_wf_list = []
tau10_wf_list = []

for c in range(10):

    with open(BASE_DIR/f"bym01_chain{c}+cov.pkl","rb") as f:
        d = pickle.load(f)
    eta01_wf_list.append(d["eta"][:, ::15])
    tau01_wf_list.append(d["tau"][:, ::15])

    with open(BASE_DIR/f"bym10_chain{c}+cov.pkl","rb") as f:
        d = pickle.load(f)
    eta10_wf_list.append(d["eta"][:, ::15])
    tau10_wf_list.append(d["tau"][:, ::15])

eta01_wf = np.concatenate(eta01_wf_list, axis=1)
tau01_wf = np.concatenate(tau01_wf_list, axis=1)
eta10_wf = np.concatenate(eta10_wf_list, axis=1)
tau10_wf = np.concatenate(tau10_wf_list, axis=1)


M = iid01.shape[1]

def sigmoid(z):
    z = np.clip(z,-30,30)
    return 1/(1+np.exp(-z))
# ONE-STEP-AHEAD SIMULATION
def simulate_one_step(get_eta):

    draws = np.arange(M)
    I_pred = np.zeros(M)

    for j,m in enumerate(tqdm(draws)):

        y_rep = np.zeros((S,T))
        y_rep[:,0] = y[:,0]

        for tt in range(1,T):

            eta01, eta10 = get_eta(m,tt-1)

            p01 = sigmoid(eta01)
            p10 = sigmoid(eta10)

            prev = y_rep[:,tt-1]

            prob = (1-prev)*p01 + prev*(1-p10)
            y_rep[:,tt] = np.random.binomial(1, prob)

        y_bar = y_rep[:,1:].mean(axis=1)
        yc = y_bar - y_bar.mean()

        I_pred[j] = (S/w_sum)*((yc @ (W @ yc))/(yc@yc))

    return I_pred
# MODEL DEFINITIONS
def eta_iid(m,t):
    e1 = sum(cov4[t,k]*iid01[k*S:(k+1)*S,m] for k in range(4))
    e2 = sum(cov4[t,k]*iid10[k*S:(k+1)*S,m] for k in range(4))
    return e1,e2


def eta_week(m,t):
    w = week[t]
    e1 = np.zeros(S)
    e2 = np.zeros(S)
    for k in range(8):
        beta1 = eta01_week[k*S:(k+1)*S,m]
        beta2 = eta10_week[k*S:(k+1)*S,m]
        scale1 = tau01_week[k*52+w,m]
        scale2 = tau10_week[k*52+w,m]
        e1 += cov8[t,k]*beta1*scale1
        e2 += cov8[t,k]*beta2*scale2
    return e1,e2


def eta_week_cov(m,t):
    w = week[t]
    e1 = np.zeros(S)
    e2 = np.zeros(S)
    for k in range(8):
        beta1 = eta01_wf[k*S:(k+1)*S,m]
        beta2 = eta10_wf[k*S:(k+1)*S,m]
        scale1 = tau01_wf[k*52+w,m]
        scale2 = tau10_wf[k*52+w,m]
        e1 += cov8[t,k]*beta1*scale1
        e2 += cov8[t,k]*beta2*scale2
    gamma1 = eta01_wf[8*S:8*S+3,m]
    gamma2 = eta10_wf[8*S:8*S+3,m]
    fac = np.column_stack([t_trend[t]*lat,
                           t_trend[t]*elev,
                           t_trend[t]*temp[:,t]])
    return e1 + fac@gamma1, e2 + fac@gamma2
# RUN ALL
print("IID"); I_iid = simulate_one_step(eta_iid)
# print("BYM"); I_bym = simulate_one_step(eta_bym)
print("BYM weekly"); I_week = simulate_one_step(eta_week)
# print("BYM+cov"); I_cov = simulate_one_step(eta_cov)
print("Weekly+cov"); I_week_cov = simulate_one_step(eta_week_cov)
# SUMMARY
def summary(name,x):
    print("\n",name)
    print("Mean:",x.mean())
    print("SD:",x.std())
    print("P(I>=I_obs):",np.mean(x>=I_obs))

summary("IID",I_iid)
# summary("BYM",I_bym)
summary("BYM weekly",I_week)
# summary("BYM+cov",I_cov)
summary("Weekly+cov",I_week_cov)
# PLOT
plt.figure(figsize=(8,5))
plt.hist(I_iid,25,alpha=.4,density=True,label="IID")
# plt.hist(I_bym,25,alpha=.4,density=True,label="BYM")
plt.hist(I_week,25,alpha=.4,density=True,label="BYM weekly")
# plt.hist(I_cov,25,alpha=.4,density=True,label="BYM+cov")
plt.hist(I_week_cov,25,alpha=.4,density=True,label="Weekly+cov")
plt.axvline(I_obs,color="black",linestyle="--",linewidth=2,label="Observed")
plt.legend()
plt.title("Aggregated Moran PPC (One-step ahead)")
plt.tight_layout()
plt.show()

In [ ]:
# OBSERVED WEEKLY MORAN
I_obs_week = np.zeros(52)

for w in range(52):
    idx = np.where(week == w)[0]
    y_bar = y[:, idx].mean(axis=1)
    yc = y_bar - y_bar.mean()
    I_obs_week[w] = (S/w_sum)*((yc @ (W @ yc))/(yc@yc))
    
    

def eta_iid(m,t):
    e1 = sum(cov4[t,k]*iid01[k*S:(k+1)*S,m] for k in range(4))
    e2 = sum(cov4[t,k]*iid10[k*S:(k+1)*S,m] for k in range(4))
    return e1,e2

def eta_week(m,t):
    w = week[t]
    e1 = np.zeros(S)
    e2 = np.zeros(S)
    for k in range(8):
        beta1 = eta01_week[k*S:(k+1)*S,m]
        beta2 = eta10_week[k*S:(k+1)*S,m]
        e1 += cov8[t,k]*beta1*tau01_week[k*52+w,m]
        e2 += cov8[t,k]*beta2*tau10_week[k*52+w,m]
    return e1,e2

def eta_week_cov(m,t):

    w = week[t]

    e1 = np.zeros(S)
    e2 = np.zeros(S)

    for k in range(8):

        beta1 = eta01_wf[k*S:(k+1)*S, m]
        beta2 = eta10_wf[k*S:(k+1)*S, m]

        scale1 = tau01_wf[k*52 + w, m]
        scale2 = tau10_wf[k*52 + w, m]

        e1 += cov8[t,k] * beta1 * scale1
        e2 += cov8[t,k] * beta2 * scale2

    # ----- global covariates -----
    gamma1 = eta01_wf[8*S:8*S+3, m]
    gamma2 = eta10_wf[8*S:8*S+3, m]

    fac = np.column_stack([
        t_trend[t]*lat,
        t_trend[t]*elev,
        t_trend[t]*temp[:,t]
    ])

    e1 += fac @ gamma1
    e2 += fac @ gamma2

    return e1, e2
# WEEKLY SIMULATION
def simulate_weekly(get_eta):

    draws = np.arange(M)
    I_pred = np.zeros((52, M))

    for j,m in enumerate(tqdm(draws)):

        y_rep = np.zeros((S,T))
        y_rep[:,0] = y[:,0]

        for tt in range(1,T):

            eta01,eta10 = get_eta(m,tt-1)

            p01 = sigmoid(eta01)
            p10 = sigmoid(eta10)

            prev = y_rep[:,tt-1]
            prob = (1-prev)*p01 + prev*(1-p10)

            y_rep[:,tt] = np.random.binomial(1,prob)

        for w in range(52):
            idx = np.where(week==w)[0]
            y_bar = y_rep[:,idx].mean(axis=1)
            yc = y_bar - y_bar.mean()
            I_pred[w,j] = (S/w_sum)*((yc@(W@yc))/(yc@yc))

    return I_pred
# RUN ALL
print("IID"); I_iid = simulate_weekly(eta_iid)
# print("BYM"); I_bym = simulate_weekly(eta_bym)
print("BYM weekly"); I_week = simulate_weekly(eta_week)
# print("BYM+cov"); I_cov = simulate_weekly(eta_cov)
print("Weekly+cov"); I_week_cov = simulate_weekly(eta_week_cov)
# SUMMARY
def summarize(I_pred):

    mean = I_pred.mean(axis=1)
    lo = np.percentile(I_pred,2.5,axis=1)
    hi = np.percentile(I_pred,97.5,axis=1)
    pval = np.mean(I_pred >= I_obs_week[:,None],axis=1)

    return mean,lo,hi,pval

res = {
    "IID": summarize(I_iid),
    # "BYM": summarize(I_bym),
    "BYM weekly": summarize(I_week),
    # "BYM+cov": summarize(I_cov),
    "Weekly+cov": summarize(I_week_cov)
}



In [ ]:
# PLOT
weeks = np.arange(1,53)

for name,(mean,lo,hi,pval) in res.items():

    plt.figure(figsize=(9,5))

    plt.plot(weeks,I_obs_week,color="black",linewidth=2,label="Observed")
    plt.plot(weeks,mean,label="Posterior mean")

    plt.fill_between(weeks,lo,hi,alpha=0.3,label="95% CI")

    plt.title(name+" – Weekly Moran PPC")
    plt.xlabel("Week")
    plt.ylabel("Moran's I")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

weeks = np.arange(1,53)

plt.figure(figsize=(10,6))

plt.plot(weeks, I_obs_week,
         color="black",
         linewidth=2.5,
         label="Observed")

colors = ["tab:blue", "tab:orange", "tab:green"]

for (name, (mean, lo, hi, pval)), color in zip(res.items(), colors):

    plt.plot(weeks, mean,
             color=color,
             linewidth=2,
             label=f"{name} mean")

    plt.fill_between(weeks, lo, hi,
                     color=color,
                     alpha=0.2)

plt.title("Weekly Moran PPC (All Models)")
plt.xlabel("Week")
plt.ylabel("Moran's I")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
models_to_check = ["IID", "BYM weekly", "Weekly+cov"]

print("\n================ CI COVERAGE ANALYSIS ================")

for name in models_to_check:

    mean, lo, hi, pval = res[name]

    cover = (I_obs_week >= lo) & (I_obs_week <= hi)

    coverage_rate = cover.sum()
    too_low = np.sum(I_obs_week < lo)
    too_high = np.sum(I_obs_week > hi)

    print("\nModel:", name)
    print("Coverage rate:", coverage_rate)
    print("Too low proportion:", too_low)
    print("Too high proportion:", too_high)

In [ ]:
def compute_weekly_pvalues(I_pred, I_obs):
    """
    I_pred : (52, Nrep)
    I_obs  : (52,)
    """
    return np.mean(I_pred >= I_obs[:, None], axis=1)
p_iid = compute_weekly_pvalues(I_iid, I_obs_week)
p_week = compute_weekly_pvalues(I_week, I_obs_week)
p_week_cov = compute_weekly_pvalues(I_week_cov, I_obs_week)

In [ ]:
weeks = np.arange(52)

plt.figure(figsize=(11,5))

colors = {
    "IID": "tab:blue",
    "BYM weekly": "tab:orange",
    "Weekly+cov": "tab:green"
}

series = {
    "IID": p_iid,
    "BYM weekly": p_week,
    "Weekly+cov": p_week_cov
}

# Main curves
for name, values in series.items():
    plt.plot(
        weeks,
        values,
        color=colors[name],
        linewidth=1.6,
        marker="o",
        markersize=4,
        alpha=0.9,
        label=name
    )

# Lower and thinner bands
band_height = 0.010
band_positions = {
    "IID": -0.070,
    "BYM weekly": -0.055,
    "Weekly+cov": -0.040
}

for name, values in series.items():

    mask = (values < 0.05) | (values > 0.95)
    y0 = band_positions[name]

    for w in weeks[mask]:
        plt.gca().add_patch(
            plt.Rectangle(
                (w - 0.4, y0),
                0.8,
                band_height,
                color=colors[name],
                alpha=0.85
            )
        )

plt.axhline(0.05, linestyle="--", color="red", alpha=0.6)
plt.axhline(0.95, linestyle="--", color="red", alpha=0.6)
plt.axhline(0.5, linestyle="--", color="gray", alpha=0.6)
ax = plt.gca()


plt.ylim(-0.085, 1.05)
plt.yticks(np.arange(0, 1.01, 0.1))

plt.xlabel("Week")
plt.ylabel("Posterior predictive p-value")
plt.title("Weekly Moran's I PPC p-values")

plt.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.22),
    ncol=3,
    frameon=False,
    fontsize=9
)

plt.subplots_adjust(bottom=0.32)
plt.tight_layout()
plt.show()